### Group info

ID: G5_4

Members:
- Vinicius Millan Santa Catharina
- Zaur Kambiev
- Stig Espen Paulsberg

### Imports

### Task 1

### Task 2

### Task 3

Define a function for handling price outliers.

In [ ]:
def handle_outliers(prices: ArrayLike) -> NDArray[np.float64]:
    """Replace price outliers with the median of valid prices.

    Outliers are defined as prices more than three standard deviations
    away from the mean.

    Parameters
    ----------
    prices : array_like
        Prices convertible to a 1D NumPy array of floats.

    Returns
    -------
    ndarray
        A 1D NumPy array of prices with outliers replaced by the
        median of valid prices.
    """

    prices = np.asarray(prices, dtype=np.float64)

    if prices.ndim != 1:
        raise ValueError("Prices must be 1D.")

    mean = prices.mean()
    std = prices.std()

    valid_mask = abs(prices - mean) <= 3 * std

    median = np.median(prices[valid_mask])

    prices[~valid_mask] = median

    return prices

Define a function for calculating the refund ratio of items.

In [ ]:
EPS = 1e-6


def compute_refund_ratio(
        sales_count: NDArray,
        refund_count: NDArray
) -> NDArray:
    """Compute the refund-to-sale ratio for each item.

    A small positive constant (EPS) is added to the number of sales
    to prevent division by zero.

    Parameters
    ----------
    sales_count : ndarray
        Number of sales for each item.
    refund_count : ndarray
        Number of refunds for each item.

    Returns
    -------
    ndarray
        An array containing element-wise refund-to-sale ratios.
    """

    return refund_count / (sales_count + EPS)

### Task 4

Define a function that converts a list of transactions into a `pandas.DataFrame`.

In [ ]:
def list_to_df(transactions: list[dict]) -> pd.DataFrame:
    """Convert a list of transactions into a pandas DataFrame.

    Convert the timestamp column to datetime, set it as the DataFrame index,
    and sort the transactions chronologically.

    Parameters
    ----------
    transactions : list[dict]
        A list of dictionaries containing a `timestamp` field
        in YYYY-MM-DD HH:MM:SS format.

    Returns
    -------
    pd.DataFrame
        A DataFrame indexed by timestamp, sorted in chronological order.
    """

    transactions = pd.DataFrame(transactions)

    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

    transactions = transactions.set_index("timestamp").sort_index()

    return transactions

Define a function that resamples transactions into 1-hour intervals. 

In [ ]:
def resample_hourly(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate transaction prices into hourly intervals by item.

    We group transactions only by item and ignore all differences
    in other columns (e.g. currency, status).

    Parameters
    ----------
    transactions : pd.DataFrame
        Transaction records with a DatetimeIndex and columns for item, price.

    Returns
    -------
    pd.DataFrame
        Hourly price totals grouped by item.
    """

    return (
        transactions
        .groupby("item")
        .resample("h")
        .agg(total_price=("price", "sum"))
    )

Define a function to correct price values for NOK transactions.

In [ ]:
def correct_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Correct the price glitch in NOK transactions."""
    
    result = transactions.copy()
    result.loc[result['currency'] == 'NOK', 'price']*=10

    return result

Define a function to add a new column containing prices converted to NOK.

In [ ]:
def add_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Add a column with transaction prices converted to NOK."""

    conversion_rates = {
        "NOK": 1.0,
        "EUR": 10.4,
        "USD": 9.3
    }

    result = transactions.copy()
    result['nok_price'] = (result['currency'].map(conversion_rates)*result['price']).round(2)

    return result

Define a function to create a pivot table displaying total transaction revenue grouped by item across status categories.

In [ ]:
def transaction_revenue_pivot(transactions: pd.DataFrame) -> pd.DataFrame:

    transactions_pivot = pd.pivot_table(
            data=transactions,
            values='nok_price',
            index='item',
            columns='status',
            aggfunc='sum'
    )

    return transactions_pivot

### Task 5